# **Part 1: Mapping Exploration Questions to Visualization**


**1.   What countries have been the leading producers of tea over the period of time?**

*Chart Type: Multi-line chart* <br>
*Number of Series: Multiple*
- Since tea production is time series data (continuous values measured over time, specifically years) and comparing multiple regions (which is categorical), a multi-line chart shows trends over time pretty well with regional comparsions. It also makes it easy to identify growth rates, shifts in production dominance, and long-term leaders

<br>

**2.   How has tea production changed over time across different regions?**

*Chart Type: Stacked Area Chart* <br>
*Number of Series: Multiple*
- This question talks about regional contributions over time, and the data includes continuous production values grouped by areas/regions. We believe that a stacked area fits best as it shows the time series data but also showcases each region's contribution to global production. This makes it more intutitive to gauge shifts in regions over time periods

<br>


**3. How does tea compare nutritionally to other low-calorie drinks?**

*Chart Type: Radar Chart* <br>
*Number of Series: Multiple*
- Nutirional metrics such as calories, caffiene, sugar are multiple continuous variables. Since we are comparing a small set of beverages across multiple quantitative variables, a radar chart works really well for this :) After researching more, we also found that radar charts can also show different stregnths and weaknesses across nutrients in a pretty compact format

<br>

**4. What is the overall nutritional profile of tea (overall)?**

*Chart Type: Bar Chart* <br>
*Number of Series: One*
- Since we are presenting average nutritional values here, a bar chart is appropriate because it shows magnitudes for a category (tea). I think that if multiple tea types of samples were included, a boxplot would be better represented. However, since we are only exploring one category, a bar chart would be good

<br>

**5. To what extent is tea production in 1990 correlate with tea production in 2021?**

*Chart Type: Scatterplot* <br>
*Number of Series: Two continuous variables*
- Tea production in 1990 and 2021 are both continous variables. We found that a scatterplot is the most effective way to visualize the correlation between past output and present day results

<br>

In [ ]:
import altair as alt

alt.renderers.enable('html')

line_chart.save('line_chart.html')

# **Part 2: Creating Visualizations**

In [ ]:
# Getting data (got this from previous notebooks)
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

tea_p = pd.read_csv('/content/sample_data/Tea_crops_primary_production_quantity (1).csv')
nutrition = pd.read_csv('/content/sample_data/daily_food_nutrition_dataset (1).csv', on_bad_lines='skip')

nutrition = nutrition.rename(columns={
    'Calories (kcal)': 'Calories',
    'Protein (g)': 'Protein',
    'Carbohydrates (g)': 'Carbs',
    'Fat (g)': 'Fat',
    'Sugars (g)': 'Sugars'
})

tea_p = tea_p.rename(columns={'Area': 'Country',
    'Item': 'Crop'
})
print("Datasets loaded and columns renamed :)")

Datasets loaded and columns renamed :)


In [ ]:
import altair as alt

# Melting the data frame before analysis done
tea_long = tea_p.melt(
    id_vars = 'Country',
    var_name = 'Year',
    value_name = 'Quantity',
)

tea_long['Year'] = tea_long['Year'].astype(int)
tea_long.head()

,Country,Year,Quantity
0,Argentina,1990,50505.0
1,Azerbaijan,1990,NaN
2,Bangladesh,1990,39075.0
3,Bolivia (Plurinational State of),1990,500.0
4,Brazil,1990,9813.0


In [ ]:
# Research question: "What countries have been the leading producers of tea over the period of time?"
# Multi-Line Chart

# Consolidating both Chinas and getting the top 5 producing countries
tea_clean = tea_long[tea_long["Country"] != 'China']
top_countries = tea_clean.groupby('Country')['Quantity'].sum().nlargest(5).index
tea_top = tea_clean[tea_clean['Country'].isin(top_countries)]

# Plotting
line_chart = alt.Chart(tea_top).mark_line(point=True).encode(
    x = alt.X('Year:T', title='Year'),
    y = alt.Y('Quantity:Q', title='Production Quantity (Tonnes)'),
    color='Country:N',
    tooltip=['Country', 'Year', 'Quantity']
).properties(
    width = 600,
    height = 400,
    title='Top 5 Tea Producing Countries Over Time (1990-2021)'
).interactive()

line_chart

alt.Chart(...)

For the research question, "What countries have been the leading producers of tea over the period of time?", using the data visualization we can deduce that from 1990 and 2021, China (mainland) was a top competitor with India to the global leader of tea with growth that pretty much outpaced all other regions. In second place, we see India. The production gap also widened between Kenya, Sri Lanka, and Turkiye but remained in lower volumes than China and India.

In [ ]:
# Research Question: "How has tea production changed over time across different regions?
# Stacked Area Chart

# Consolidating both Chinas
tea_long['Country'] = tea_long['Country'].replace('China, mainland', 'China')
tea_long = tea_long.groupby(['Country', 'Year'], as_index=False)['Quantity'].sum()

# Getting 10 countries (largest)
top_countries = tea_long.groupby('Country')['Quantity'].sum().nlargest(10).index
tea_long['Country_Grouped'] = tea_long['Country'].apply(lambda x: x if x in top_countries else "Other")

# Plotting
area_chart = alt.Chart(tea_long).mark_area(opacity=0.6).encode(
    x = alt.X('Year:O', title='Year'),
    y = alt.Y('Quantity:Q', stack='zero', axis = alt.Axis(format="~s")),
    color=alt.Color('Country_Grouped:N', scale=alt.Scale(scheme='set1')),
    tooltip=['Country', 'Year', 'Quantity']
).properties(
    width=600,
    height=400,
    title='Tea Production by Country over Time'
).interactive()

area_chart

alt.Chart(...)

For the research question, "How has tea production changed over time across different regions?", global tea production experienced massive growth between 1990 and 2021 by tripling in volume from appx 15 million to 40 million units. This growth was mainly by China. Regions like India and Kenya showed moderate steady increases and traditional producers like Sri Lanka and Turkiye maintained flat production levels.

In [ ]:
# Research Question: "How does tea compare nutritionally to other low-calorie drinks?"
# Radial Chart

# Used plotly for this instead of Altair
import plotly.express as px

keywords = ['Coffee (black)', 'Iced Tea (Sweetened 16oz)', 'Diet Soda (1 can)', 'Orange Juice (1 cup)']
nutr_df = nutrition[nutrition['Food_Item'].isin(keywords)]

nutr_long = nutr_df.melt(
    id_vars = "Food_Item",
    value_vars = nutrients,
    var_name = 'Nutrient',
    value_name = 'Value',
)

figure = px.line_polar(nutr_long, r='Value', theta='Nutrient', color="Food_Item", line_close=True, template='plotly_dark', title="Nutritional Comparsion: Tea vs Other Drinks")


# Using log scaling (researched a bit on this lol) to start normalizing :)
figure.update_polars(radialaxis=dict(visible=True, range=[0, nutr_long['Value'].max() * 1.1]))

figure.update_traces(fill="toself", opacity=0.5)
figure.update_traces(mode="lines+markers", marker=dict(size=8))

figure.show()

For the research question, "How does tea compare nutritionally to other low-calorie drinks?", based on the radial chart, we found that sweetened iced tea stands out as higher in both calories (180) and sugars compared to coffe and diet soda, which remain near zero or close to zero. I also observed that orange juice occupies a middle ground for calories and sugar and all four drinks show no or very little amounts for protein and fat.

In [ ]:
# Research Question: "What is the overall nutritional profile of tea (overall)?"
# Bar Chart

# Filtering the tea data from nutrition dataset
tea_nutrition = nutrition[nutrition['Food_Item'].str.contains('Tea', case=False)]

# Average nutritional values
tea_avg = tea_nutrition[['Calories', 'Protein', 'Carbs', 'Fat', 'Sugars']].mean().reset_index()
tea_avg.columns = ['Nutrient', 'Value']

# Getting macronutrinets only
tea_macros_only = tea_avg[tea_avg['Nutrient'] != 'Calories']

# Plotting
bar_chart = alt.Chart(tea_macros_only).mark_bar(color='purple').encode(
    x=alt.X("Nutrient:N", title='Nutrient', sort="-y"),
    y=alt.Y("Value:Q", title="Average Value (grams)"),
    tooltip = ['Nutrient', 'Value']
).properties(
    title='Average Macronutrient Profile of Tea (grams)'
)

bar_chart

alt.Chart(...)

For the research question, ""What is the overall nutritional profile of tea (overall)?" based on the bar chart, I see that carbohydrates are the most prevalent macronutrient in the profile, at around 18 grams, followed by sugars, protein and then fat. Sugars also make up about 3/4 (75%) of the profile which suggests that the beverage is pretty sweet. We also saw that it has a decent amount of protein ~ 12 grams and minimial fat.

In [ ]:
# Research Question: "To what extent does tea production in 1990 correlate to tea production in 2021?"
# Scatterplot

import pandas as pd
import altair as alt

early_year = 1990
latest_year = 2021

# Getting the years
tea_1990 = tea_long[tea_long['Year'] == early_year][['Country', 'Quantity']]
tea_2021 = tea_long[tea_long['Year'] == latest_year][['Country', 'Quantity']]

tea_1990 = tea_1990.rename(columns={'Quantity': 'Production_1990'})
tea_2021 = tea_2021.rename(columns={'Quantity': 'Production_2021'})

# Merging
tea_growth = pd.merge(tea_1990, tea_2021, on="Country")

# Plotting
scatter = alt.Chart(tea_growth).mark_circle(size=100, opacity=0.7).encode(
    x=alt.X('Production_1990:Q', title='Tea Production in 1990', scale=alt.Scale(domain=[0, tea_growth['Production_1990'].max()], nice=True, zero=True)),
    y=alt.Y('Production_2021:Q', title='Tea Production in 2021', scale=alt.Scale(domain=[0, tea_growth['Production_2021'].max()], nice=True,  zero=True)),
    tooltip=['Country', 'Production_1990', 'Production_2021']
).properties(
    width = 600,
    height= 400,
    title='Tea Production Growth: 1990 vs 2021'
).interactive()

scatter

alt.Chart(...)

For the research question, "To what extent does tea production in 1990 correlate to tea production in 2021?" we used a scatterplot. The scatterplot shows a strong positive correlation between 1990 and 2021 tea production data, indicating that countries with established industries 30 years back are the dominant global producers today as well. This association is sort of linear, with some outliers as well

## **Part 3: Analytical and Statistical Methods**

Domain specific analytical methods found at https://www.statsmodels.org/stable/tsa.html. A few candidates are listed below.

stattools.acovf: Estimate autocovariances.

stattools.acf: Calculate the autocorrelation function.

stattools.pacf_burg: Calculate Burg"s partial autocorrelation estimator.

stattools.adfuller: Augmented Dickey-Fuller unit root test.


## Method 1: Autocorrelation Function

ACF can be used to check whether past production values will correlate with future ones.

In [ ]:
#Importing methods for time-series data
from statsmodels.tsa.stattools import acf, adfuller

#Working with China's data because it's the highest producer
china = (
    tea_long[tea_long['Country'] == 'China']
    .set_index('Year')['Quantity']
    .sort_index()
)

In [ ]:
acf_values = acf(
    china, adjusted=False, nlags=None, qstat=False, fft=True, alpha=None, bartlett_confint=True, missing='none' )

print("ACF values:", acf_values)

ACF values: [ 1.          0.87962791  0.78679344  0.69385502  0.60468027  0.51569378
  0.42829571  0.3324306   0.24135108  0.15787106  0.07770921  0.00404514
 -0.06086371 -0.12173501 -0.17933507 -0.2321873 ]


Overall, China's tea production is strongly correlated with it's past values. The ACF value being very high at lag 1 indicates that China's tea production for this year is strongly predicted by last year's. Since the ACF value gradually decreases, it indicates that the further back you go, the weaker the relationship. It's clear that China's tea production has been increasing yearly.

### Method 2: Augmented Dickey-Fuller Test

The ADF test can be used to see whther the series has a stable mean over time or if it's trending upwards/downwards by testing for a unit root.

In [ ]:
adf_result = adfuller(china, maxlag=None, regression='c', autolag='AIC', store=False, regresults=False)

print('ADF Statistic: ', adf_result[0])
print('p-value: ', adf_result[1])

ADF Statistic:  2.421868554712289
p-value:  0.9990202323408395


Since the p value is above .05, we fail to reject the null hypothesis, meaning that there is a unit root.This means that the data does not stay around a fixed average, meaning that it is either increasing or decreasing. Based on what we learned through the previous statistical analysis, China's production is increasing, which is why the data does not remain near a constant average.